In [ ]:
from soja_clima.inmet import copiar_um_arquivo, agregar_diario
from soja_clima.inmet import COLUNAS_CONHECIDAS, COLUNAS_ESSENCIAIS

In [ ]:
# ============================================================
# CÉLULA 1 — Conectar ao Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Pasta com as subpastas por ano (ex: .../2015/, .../2016/, ...)
PASTA_RAIZ = "/content/drive/MyDrive/PCD2/DadosClimaticos"


# ============================================================
# CÉLULA 2 — Imports
# ============================================================
import pandas as pd
import numpy as np
import glob
import os
import shutil
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm


# ============================================================
# CÉLULA 3 — Copiar os dados do Drive para o disco local (em paralelo)
# ============================================================
# Com muitos arquivos, ler direto do Drive via FUSE é lento e instável
# (falhas intermitentes tipo "No such file or directory"). Copiar em
# paralelo para o disco local (/content/) resolve os dois problemas:
# velocidade e confiabilidade. Cada arquivo tem retry individual.

PASTA_LOCAL = "/content/dados_inmet"

def copiar_um_arquivo(origem, destino, tentativas=3):
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    for _ in range(tentativas):
        try:
            shutil.copy2(origem, destino)
            return True
        except Exception:
            time.sleep(2)
    return False

arquivos_origem = sorted(set(
    glob.glob(f"{PASTA_RAIZ}/**/*.CSV", recursive=True) +
    glob.glob(f"{PASTA_RAIZ}/**/*.csv", recursive=True)
))
print(f"{len(arquivos_origem)} arquivos encontrados no Drive.")

pares = [(o, o.replace(PASTA_RAIZ, PASTA_LOCAL)) for o in arquivos_origem]

falhas_copia = []
with ThreadPoolExecutor(max_workers=16) as executor:
    futuros = {executor.submit(copiar_um_arquivo, o, d): o for o, d in pares}
    for fut in tqdm(as_completed(futuros), total=len(futuros), desc="Copiando"):
        if not fut.result():
            falhas_copia.append(futuros[fut])

print(f"Cópia concluída. {len(falhas_copia)} falha(s) de cópia.")
for f in falhas_copia[:20]:
    print(f"  - {f}")


# ============================================================
# CÉLULA 4 — Leitura e limpeza de um arquivo INMET
# ============================================================
def normalizar_chave_meta(chave):
    """A partir de 2019 o INMET passou a exportar alguns cabeçalhos com
    caracteres corrompidos (ex: 'ESTAC?O' em vez de 'ESTAÇÃO'). Normaliza
    as variantes conhecidas para uma chave única."""
    chave = chave.strip().upper()
    correcoes = {
        'REGI?O': 'REGIAO', 'REGIÃO': 'REGIAO',
        'ESTAC?O': 'ESTACAO', 'ESTAÇÃO': 'ESTACAO',
        'DATA DE FUNDAC?O': 'DATA DE FUNDACAO', 'DATA DE FUNDAÇÃO': 'DATA DE FUNDACAO',
    }
    return correcoes.get(chave, chave)


# Nomes de coluna em maiúsculo -> nome canônico usado no resto do pipeline.
# A comparação é sempre feita em maiúsculo (ver ler_arquivo_inmet), o que já
# resolve variações como 'RADIACAO GLOBAL (KJ/m²)' (2015/2019) vs
# 'RADIACAO GLOBAL (Kj/m²)' (2020+) sem precisar listar as duas formas.
COLUNAS_CONHECIDAS = {
    'PRECIPITAÇÃO TOTAL, HORÁRIO (MM)': 'precipitacao',
    'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'.upper(): 'temperatura',
    'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)'.upper(): 'temp_max_hora',
    'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)'.upper(): 'temp_min_hora',
    'UMIDADE RELATIVA DO AR, HORARIA (%)': 'umidade',
    'RADIACAO GLOBAL (KJ/M²)': 'radiacao',
    'VENTO, VELOCIDADE HORARIA (M/S)': 'vento',
}

COLUNAS_ESSENCIAIS = {'data', 'hora', 'precipitacao', 'temperatura', 'temp_max_hora',
                       'temp_min_hora', 'umidade', 'radiacao', 'vento'}


def ler_arquivo_inmet(path):
    """Lê um único CSV do INMET (formato padrão de estação automática)
    e retorna um dataframe horário limpo, com metadados da estação.
    Compatível com o formato antigo (até ~2018) e os formatos 2019+, que
    mudaram nomes de coluna (inclusive maiúscula/minúscula) e corromperam
    alguns cabeçalhos de metadados. Levanta erro claro se alguma coluna
    essencial não for encontrada, em vez de falhar mais adiante."""
    meta = {}
    with open(path, encoding='latin1') as f:
        for _ in range(8):
            linha = f.readline().strip()
            chave, valor = linha.split(':;')
            meta[normalizar_chave_meta(chave)] = valor.strip()

    df = pd.read_csv(path, sep=';', encoding='latin1', skiprows=8, decimal=',')
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]  # remove coluna fantasma do ';' final

    mapa_renomeacao = {}
    for col in df.columns:
        col_upper = col.strip().upper()
        if col_upper.startswith('DATA'):
            mapa_renomeacao[col] = 'data'
        elif col_upper.startswith('HORA'):
            mapa_renomeacao[col] = 'hora'
        elif col_upper in COLUNAS_CONHECIDAS:
            mapa_renomeacao[col] = COLUNAS_CONHECIDAS[col_upper]

    df = df.rename(columns=mapa_renomeacao)

    faltando = COLUNAS_ESSENCIAIS - set(df.columns)
    if faltando:
        raise ValueError(f"Colunas essenciais não encontradas em {path}: {faltando}")

    df['data'] = pd.to_datetime(df['data'])

    colunas_numericas = df.select_dtypes(include=[np.number]).columns
    df[colunas_numericas] = df[colunas_numericas].replace(-9999, np.nan)

    # Radiação: -9999 à noite geralmente significa 0, não ausência real
    hora_num = df['hora'].astype(str).str.strip().str.slice(0, 2).astype(int)
    noite = (hora_num < 9) | (hora_num > 19)
    df.loc[df['radiacao'].isna() & noite, 'radiacao'] = 0
    df['radiacao'] = df['radiacao'].clip(lower=0)

    df['codigo_estacao'] = meta['CODIGO (WMO)']
    df['uf']             = meta['UF']
    df['estacao']        = meta['ESTACAO']
    df['latitude']       = float(meta['LATITUDE'].replace(',', '.'))
    df['longitude']      = float(meta['LONGITUDE'].replace(',', '.'))

    return df


# ============================================================
# CÉLULA 5 — Agregação hora -> dia, arquivo por arquivo (streaming)
# ============================================================
# Nunca concatenamos os dados horários de todos os arquivos de uma vez
# (seriam dezenas de milhões de linhas). Cada arquivo é processado e
# reduzido para diário individualmente; só os diários são acumulados.

def agregar_diario(path):
    df = ler_arquivo_inmet(path)

    diario = df.groupby(
        ['codigo_estacao', 'uf', 'estacao', 'latitude', 'longitude', 'data']
    ).agg(
        prec_total=('precipitacao', 'sum'),
        temp_media=('temperatura', 'mean'),
        temp_max=('temp_max_hora', 'max'),
        temp_min=('temp_min_hora', 'min'),
        umid_media=('umidade', 'mean'),
        radiacao_total=('radiacao', 'sum'),
        vento_medio=('vento', 'mean'),
        n_horas_validas=('temperatura', 'count'),
    ).reset_index()

    return diario

arquivos_locais = sorted(set(
    glob.glob(f"{PASTA_LOCAL}/**/*.CSV", recursive=True) +
    glob.glob(f"{PASTA_LOCAL}/**/*.csv", recursive=True)
))

diarios = []
erros_processamento = []
for path in tqdm(arquivos_locais, desc="Processando"):
    try:
        diarios.append(agregar_diario(path))
    except Exception as e:
        erros_processamento.append((path, str(e)))

print(f"\n{len(erros_processamento)} arquivo(s) com falha no processamento.")
for path, msg in erros_processamento[:20]:
    print(f"  - {path}: {msg}")

if not diarios:
    raise RuntimeError("Nenhum arquivo foi processado com sucesso — confira os erros acima.")

diario = pd.concat(diarios, ignore_index=True)
diario['amplitude_termica'] = diario['temp_max'] - diario['temp_min']
diario['dia_seco'] = diario['prec_total'] < 1
diario['gdd'] = ((diario['temp_max'] + diario['temp_min']) / 2 - 10).clip(lower=0)

# Descarta dias com muitas horas faltando (controle de qualidade)
diario = diario[diario['n_horas_validas'] >= 18].copy()

print("\nShape final (diário):", diario.shape)
print(diario['codigo_estacao'].nunique(), "estações únicas")
print("Período coberto:", diario['data'].min(), "a", diario['data'].max())


# ============================================================
# CÉLULA 6 — Salvar o diário no Drive
# ============================================================
# Por enquanto salvamos só até o nível diário. A agregação para
# safra/fase fenológica e o merge com o SIDRA ficam para uma etapa
# seguinte, lendo este arquivo como ponto de partida.
pasta_saida = "/content/drive/MyDrive/PCD2/DadosClimaticos/processado"
os.makedirs(pasta_saida, exist_ok=True)

caminho_saida = f"{pasta_saida}/clima_diario.parquet"
diario.to_parquet(caminho_saida, index=False)
print(f"Salvo em {caminho_saida}")

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

df = pd.read_parquet("/content/drive/MyDrive/PCD2/DadosClimaticos/processado/clima_diario.parquet")
df